# Adult Census Income: baseline и обучение моделей

В этом notebook создаём baseline и обучаем основные модели.

Что входит в этот этап:

- `DummyClassifier(strategy="most_frequent")` как baseline;
- `LogisticRegression`;
- `KNeighborsClassifier`;
- `DecisionTreeClassifier`;
- `RandomForestClassifier`;
- `GradientBoostingClassifier`;
- общий `Pipeline` для каждой модели: preprocessing + model.

Cross-validation, GridSearchCV, финальная оценка на test set и анализ ошибок здесь не выполняются. Эти этапы остаются для следующей части проекта.

## Импорты и настройки

In [1]:
import os

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

import sys
from pathlib import Path

sys.path.append(str(Path.cwd()))
sys.path.append(str(Path.cwd() / "src"))

from adult_income_utils import (
    RANDOM_STATE,
    add_features,
    get_feature_lists,
    load_clean_adult_data,
    make_X_y,
    make_preprocessor,
)

pd.set_option("display.max_columns", 100)

## Подготовка train/test split

Используем ту же подготовку данных, что и в предыдущем notebook:

- загрузка очищенных данных;
- feature engineering;
- разделение на `X` и `y`;
- `train_test_split` со `stratify=y`.

Test set создаётся, но в этом notebook не используется для оценки качества.

In [2]:
df_clean = load_clean_adult_data()
df_features = add_features(df_clean)

X, y = make_X_y(df_features)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (39032, 21)
X_test shape: (9758, 21)
y_train shape: (39032,)
y_test shape: (9758,)


## Preprocessing

Для каждой модели будем использовать один и тот же preprocessing:

- числовые признаки: `SimpleImputer(strategy="median")` + `StandardScaler`;
- категориальные признаки: `SimpleImputer(strategy="constant", fill_value="Unknown")` + `OneHotEncoder(handle_unknown="ignore")`.

Preprocessing создаётся через `ColumnTransformer` в функции `make_preprocessor`.

In [3]:
numerical_features, categorical_features = get_feature_lists(X_train)
preprocessor = make_preprocessor(numerical_features, categorical_features)

print("Numerical features:", numerical_features)
print("\nCategorical features:", categorical_features)

Numerical features: ['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week', 'is_married', 'capital_net', 'has_capital_gain', 'has_capital_loss']

Categorical features: ['workclass', 'education', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'native_country', 'native_country_group', 'workclass_group', 'hours_per_week_bin']


## Baseline

В качестве baseline используем `DummyClassifier(strategy="most_frequent")`.

Эта модель не использует признаки `X`. Она всегда предсказывает самый частый класс в `y_train`. Для Adult Census Income это важная нижняя планка, потому что классы несбалансированы.

In [4]:
baseline = DummyClassifier(strategy="most_frequent")

baseline_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", baseline),
])

baseline_pipeline.fit(X_train, y_train)

print("Baseline fitted:", baseline_pipeline.named_steps["model"])

Baseline fitted: DummyClassifier(strategy='most_frequent')


## Основные модели

Создаём модели из классического ML-стека курса. Все модели будут обучаться на одном и том же train set и с одним и тем же preprocessing pipeline.

In [5]:
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
    ),
    "KNN": KNeighborsClassifier(),
    "DecisionTree": DecisionTreeClassifier(
        random_state=RANDOM_STATE,
        class_weight="balanced",
    ),
    "RandomForest": RandomForestClassifier(
        random_state=RANDOM_STATE,
        class_weight="balanced",
    ),
    "GradientBoosting": GradientBoostingClassifier(
        random_state=RANDOM_STATE,
    ),
}

models

{'LogisticRegression': LogisticRegression(class_weight='balanced', max_iter=1000),
 'KNN': KNeighborsClassifier(),
 'DecisionTree': DecisionTreeClassifier(class_weight='balanced', random_state=42),
 'RandomForest': RandomForestClassifier(class_weight='balanced', random_state=42),
 'GradientBoosting': GradientBoostingClassifier(random_state=42)}

## Обучение моделей

Для каждой модели создаём отдельный `Pipeline`:

```python
Pipeline([
    ("preprocessor", preprocessor),
    ("model", model),
])
```

Так preprocessing обучается только на train set и не создаёт leakage.

In [6]:
fitted_pipelines = {
    "DummyClassifier": baseline_pipeline,
}

for model_name, model in models.items():
    print(f"Training {model_name}...")

    pipeline = Pipeline(steps=[
        ("preprocessor", make_preprocessor(numerical_features, categorical_features)),
        ("model", model),
    ])

    pipeline.fit(X_train, y_train)
    fitted_pipelines[model_name] = pipeline

print("Training finished.")

Training LogisticRegression...
Training KNN...
Training DecisionTree...
Training RandomForest...
Training GradientBoosting...
Training finished.


In [7]:
list(fitted_pipelines.keys())

['DummyClassifier',
 'LogisticRegression',
 'KNN',
 'DecisionTree',
 'RandomForest',
 'GradientBoosting']

## Sanity check

Проверим только то, что каждая обученная модель технически умеет делать предсказания. Метрики качества здесь намеренно не считаем: сравнение моделей будет отдельным этапом.

In [9]:
sample_predictions = pd.DataFrame(index=X_train.head(5).index)

for model_name, pipeline in fitted_pipelines.items():
    sample_predictions[model_name] = pipeline.predict(X_train.head(5))

sample_predictions

,DummyClassifier,LogisticRegression,KNN,DecisionTree,RandomForest,GradientBoosting
12835,0,1,0,1,1,0
12131,0,0,0,0,0,0
41070,0,0,0,0,0,0
25088,0,0,0,0,0,0
3964,0,1,1,1,1,1


## Результат этапа

На этом этапе подготовлены и обучены:

- baseline `DummyClassifier`;
- `LogisticRegression`;
- `KNN`;
- `DecisionTree`;
- `RandomForest`;
- `GradientBoosting`.

Следующие этапы для продолжения проекта:

- cross-validation;
- сравнение моделей по нескольким метрикам;
- GridSearchCV для лучших моделей;
- финальная оценка на test set.